In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.feature_selection import SelectKBest,f_classif
from sklearn.feature_extraction import FeatureHasher
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("Train.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10999, 12)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


In [3]:
if 'ID' in df.columns:
    df = df.drop('ID',axis=1)
targer_col = 'Reached.on.Time_Y.N'
x = df.drop(targer_col, axis =1)
y = df[targer_col]

print(f"Features Shape: {x.head()}")
print(f"Target Shape: {y.tail()}")

Features Shape:   Warehouse_block Mode_of_Shipment  Customer_care_calls  Customer_rating  \
0               D           Flight                    4                2   
1               F           Flight                    4                5   
2               A           Flight                    2                2   
3               B           Flight                    3                3   
4               C           Flight                    2                2   

   Cost_of_the_Product  Prior_purchases Product_importance Gender  \
0                  177                3                low      F   
1                  216                2                low      M   
2                  183                4                low      M   
3                  176                4             medium      M   
4                  184                3             medium      F   

   Discount_offered  Weight_in_gms  
0                44           1233  
1                59           3088  
2

In [4]:
x.head()

,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms
0,D,Flight,4,2,177,3,low,F,44,1233
1,F,Flight,4,5,216,2,low,M,59,3088
2,A,Flight,2,2,183,4,low,M,48,3374
3,B,Flight,3,3,176,4,medium,M,10,1177
4,C,Flight,2,2,184,3,medium,F,46,2484


In [5]:
#ordinal feature encoding
importance_mapping={
    'low':1,
    'medium':2,
    'high':3
}
x['Product_importance']=x['Product_importance'].map(importance_mapping)
x['Product_importance'].head()

0    1
1    1
2    1
3    2
4    2
Name: Product_importance, dtype: int64

In [6]:
x['Gender'].unique()

array(['F', 'M'], dtype=object)

In [7]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
x['Gender'] = encoder.fit_transform(
    x['Gender'])

In [8]:
x['Gender'].head()

0    0
1    1
2    1
3    1
4    0
Name: Gender, dtype: int32

In [9]:
cols = ['Mode_of_Shipment', 'Warehouse_block']
combined = df[cols].astype(str).agg(' '.join, axis=1)
tokens = combined.apply(lambda x: x.split())
hasher = FeatureHasher(n_features=6, input_type='string')
hashed = hasher.transform(tokens)
hashed_df = pd.DataFrame(
    hashed.toarray(),
    columns=[f"hash_{i}" for i in range(6)]
)

In [10]:
x= pd.concat([x, hashed_df], axis=1)
print("New shape:", x.shape)
x.head()

New shape: (10999, 16)


,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,hash_0,hash_1,hash_2,hash_3,hash_4,hash_5
0,D,Flight,4,2,177,3,1,0,44,1233,0.0,0.0,0.0,0.0,0.0,0.0
1,F,Flight,4,5,216,2,1,1,59,3088,0.0,0.0,-2.0,0.0,0.0,0.0
2,A,Flight,2,2,183,4,1,1,48,3374,0.0,0.0,-1.0,0.0,1.0,0.0
3,B,Flight,3,3,176,4,2,1,10,1177,0.0,0.0,-2.0,0.0,0.0,0.0
4,C,Flight,2,2,184,3,2,0,46,2484,0.0,0.0,-1.0,-1.0,0.0,0.0


In [11]:
x = x.drop(['Warehouse_block', 'Mode_of_Shipment'], axis=1)

In [12]:
x.head()

,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,hash_0,hash_1,hash_2,hash_3,hash_4,hash_5
0,4,2,177,3,1,0,44,1233,0.0,0.0,0.0,0.0,0.0,0.0
1,4,5,216,2,1,1,59,3088,0.0,0.0,-2.0,0.0,0.0,0.0
2,2,2,183,4,1,1,48,3374,0.0,0.0,-1.0,0.0,1.0,0.0
3,3,3,176,4,2,1,10,1177,0.0,0.0,-2.0,0.0,0.0,0.0
4,2,2,184,3,2,0,46,2484,0.0,0.0,-1.0,-1.0,0.0,0.0


In [31]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(x)

In [13]:
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split

In [64]:
iso = IsolationForest(contamination=0.03,max_samples=100,random_state=0)
y_pred = iso.fit_predict(X_scaled)

In [65]:
df['Anomaly_IF'] = y_pred

In [66]:
print("Inliers (normal):", (y_pred == 1).sum())
print("Outliers (anomalies):", (y_pred == -1).sum())

Inliers (normal): 10669
Outliers (anomalies): 330


In [49]:
from sklearn.svm import OneClassSVM

In [50]:
ocsvm = OneClassSVM(kernel='rbf', gamma='auto', nu=0.05)
y_pred = ocsvm.fit_predict(X_scaled)
y_pred.shape

(10999,)

In [51]:
df['Anomaly'] = y_pred
print("Inliers (normal):", (y_pred == 1).sum())
print("Outliers (anomalies):", (y_pred == -1).sum())

Inliers (normal): 10450
Outliers (anomalies): 549
